# Benchmark model registry — usage guide

This notebook is the canonical walkthrough for the benchmark
**model registry**. CI executes it end-to-end on every PR
(`jupyter nbconvert --execute`), so the examples can't silently rot.

Launch jupyter from the repo root so `from benchmarks import ...` resolves
(same convention as `examples/*.ipynb`).

The registry lives in `benchmarks/registry.py`. Each model file under
`benchmarks/models/` self-registers a `ModelSpec` on import, so just touching
the `benchmarks` package populates `REGISTRY`.

## 1. Import the registry

Single entry point: `from benchmarks import REGISTRY` plus the feature / phase
constants you need for filtering.

In [ ]:
# The benchmark suite isn't shipped in the linopy wheel — it lives in-tree.
# Find the repo root by walking up from cwd and put it on sys.path so the
# import resolves whether jupyter was launched from the repo root, the
# notebooks directory, or anywhere in between.
import sys
from pathlib import Path

_p = Path.cwd()
while _p != _p.parent:
    if (_p / "benchmarks" / "registry.py").exists():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break
    _p = _p.parent

from benchmarks import (  # noqa: E402
    INTEGER,
    QUADRATIC,
    REGISTRY,
    TO_GUROBIPY,
    filter_by,
    get,
)

print(f"{len(REGISTRY)} models registered: {sorted(REGISTRY)}")

## 2. Look up one model by name

`REGISTRY[name]` returns a `ModelSpec` (frozen dataclass). Evaluating it
renders a full attribute table in Jupyter; `__repr__` gives a one-line
summary in scripts or `pytest -v` output.

In [ ]:
spec = REGISTRY["basic"]
spec

`.build(size)` constructs and returns a `linopy.Model`:

In [ ]:
spec.build(50)

`get("name")` is an equivalent functional accessor — handy when you don't
want to import `REGISTRY` directly.

In [ ]:
assert get("basic") is REGISTRY["basic"]

## 3. Iterate the whole registry

Useful when you want to sweep your own test or profiling logic across every
model — e.g. checking that a refactor didn't break any spec. Each spec's
`__repr__` carries enough info for an at-a-glance overview.

In [ ]:
list(REGISTRY.values())

## 4. Filter by feature

`filter_by(has_feature=...)` returns specs that advertise that feature. The
feature tag constants (`CONTINUOUS`, `BINARY`, `INTEGER`, `QUADRATIC`, `SOS`,
`PIECEWISE`, `MASKED`) are exported from `benchmarks`.

In [ ]:
filter_by(has_feature=QUADRATIC)

In [ ]:
filter_by(has_feature=INTEGER)

## 5. Filter by phase

Each spec declares which **phases** apply — `BUILD`, `MATRICES`, `LP_WRITE`,
`NETCDF`, `SOLVER_BUILD`, plus per-solver `TO_HIGHSPY` / `TO_GUROBIPY` /
`TO_MOSEK` / `TO_XPRESS`. Use `has_phase=` to narrow to solver-compatible
models, e.g. when writing a Gurobi-specific regression test.

In [ ]:
filter_by(has_phase=TO_GUROBIPY)

## 6. Reuse pattern — parametrize your own pytest

The pattern the suite itself uses (see `benchmarks/test_build.py` etc.) —
`iter_params(phase)` returns `(spec, size)` pairs for the given phase, and
`param_ids(...)` builds stable test IDs for `pytest.mark.parametrize`:

```python
import pytest
from benchmarks import BUILD, iter_params, param_ids

_PARAMS = iter_params(BUILD)

@pytest.mark.parametrize("spec,size", _PARAMS, ids=param_ids(_PARAMS))
def test_my_invariant(spec, size):
    m = spec.build(size)
    # ... assertion that should hold for every model
```

## 7. Reuse pattern — one-off profiling

Grab a single model at a chosen size, measure something, throw it away.
`tracemalloc` works well for in-process peak-RSS spot checks (use
`benchmarks/memory.py` + pytest-memray for the real metric).

In [ ]:
import tracemalloc  # noqa: E402

tracemalloc.start()
m = REGISTRY["sparse_network"].build(100)
_current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"sparse_network n=100: built, peak allocation ≈ {peak / 1e6:.1f} MB")
print(
    f"  {m.variables.nvars} scalar variables, {m.constraints.ncons} scalar constraints"
)

## 8. Running the benchmark suite

Everything is exposed through a typer CLI; its auto-generated help is the
source of truth:

```bash
python -m benchmarks --help            # top-level menu
python -m benchmarks run --help        # per-command flags
```

Three size tiers configured per-spec via `quick_threshold` /
`long_threshold`: `smoke` (≤ quick), `run` (≤ long), `run --long` (all
sizes). Pytest still works directly for power users.

## 9. Adding a new model

1. Drop `benchmarks/models/<name>.py` with a `build_<name>(size) -> Model`.
2. Build a `ModelSpec` and call `register(...)` at module scope. Declare
   realistic `quick_threshold` / `long_threshold` so the smoke stays fast.
3. Add an import in `benchmarks/models/__init__.py` so registration fires.

That's it — every phase test picks the spec up automatically through
`iter_params(phase)`.